# Error Handling & Edge Cases

qb-compiler uses a structured exception hierarchy rooted at
`QBCompilerError`.  All public exceptions inherit from this base class,
so callers can catch the base for blanket handling or individual
exceptions for targeted recovery.

```
QBCompilerError
  +-- CompilationError
  |     +-- InvalidCircuitError
  +-- CalibrationError
  |     +-- CalibrationStaleError
  |     +-- CalibrationNotFoundError
  +-- BackendNotSupportedError
  +-- BudgetExceededError
```

This notebook demonstrates each exception type, shows what triggers it,
and provides patterns for robust pipeline construction.

In [1]:
from qb_compiler import (
    QBCircuit,
    QBCompiler,
    CompilerConfig,
    QBCompilerError,
    CompilationError,
    InvalidCircuitError,
    CalibrationError,
    CalibrationStaleError,
    CalibrationNotFoundError,
    BackendNotSupportedError,
    BudgetExceededError,
)

## 1. Exception hierarchy

All exceptions carry a message and an optional `detail` field for
machine-readable context.

In [2]:
# Verify the inheritance chain
assert issubclass(CompilationError, QBCompilerError)
assert issubclass(InvalidCircuitError, CompilationError)
assert issubclass(CalibrationError, QBCompilerError)
assert issubclass(CalibrationStaleError, CalibrationError)
assert issubclass(CalibrationNotFoundError, CalibrationError)
assert issubclass(BackendNotSupportedError, QBCompilerError)
assert issubclass(BudgetExceededError, QBCompilerError)

print("Exception hierarchy verified.")
print(f"\nBase class: {QBCompilerError.__name__}")
for cls in [CompilationError, InvalidCircuitError, CalibrationError,
            CalibrationStaleError, CalibrationNotFoundError,
            BackendNotSupportedError, BudgetExceededError]:
    parent = cls.__bases__[0].__name__
    print(f"  {cls.__name__} -> {parent}")

Exception hierarchy verified.

Base class: QBCompilerError
  CompilationError -> QBCompilerError
  InvalidCircuitError -> CompilationError
  CalibrationError -> QBCompilerError
  CalibrationStaleError -> CalibrationError
  CalibrationNotFoundError -> CalibrationError
  BackendNotSupportedError -> QBCompilerError
  BudgetExceededError -> QBCompilerError


## 2. InvalidCircuitError

Raised when a circuit is malformed.  Common triggers:
- Qubit index out of range
- Zero or negative qubit count
- Operations on non-existent qubits

In [3]:
# Trigger 1: qubit index out of range
try:
    circ = QBCircuit(3)
    circ.cx(0, 5)  # qubit 5 doesn't exist in a 3-qubit circuit
except InvalidCircuitError as e:
    print(f"InvalidCircuitError: {e}")

# Trigger 2: zero qubits
try:
    circ = QBCircuit(0)
except InvalidCircuitError as e:
    print(f"InvalidCircuitError: {e}")

# Trigger 3: negative qubit index
try:
    circ = QBCircuit(3)
    circ.h(-1)
except InvalidCircuitError as e:
    print(f"InvalidCircuitError: {e}")

InvalidCircuitError: Qubit index 5 out of range for 3-qubit circuit
InvalidCircuitError: n_qubits must be >= 1, got 0
InvalidCircuitError: Qubit index -1 out of range for 3-qubit circuit


### The `gate` attribute

`InvalidCircuitError` has an optional `gate` attribute that identifies
which gate triggered the error (when applicable) and a `detail` field.

In [4]:
# Construct directly to see the gate and detail attributes
err = InvalidCircuitError("Unsupported gate 'foo'", gate="foo")
print(f"Message: {err}")
print(f"Gate:    {err.gate}")
print(f"Detail:  {err.detail}")

Message: Unsupported gate 'foo'
Gate:    foo
Detail:  unsupported gate: foo


## 3. BackendNotSupportedError

Raised when a requested backend name is not in the known configuration.
The exception includes the list of available backends.

In [5]:
try:
    config = CompilerConfig(backend="ibm_quantum_2000")
except BackendNotSupportedError as e:
    print(f"BackendNotSupportedError: {e}")
    print(f"\nRequested backend: {e.backend}")
    print(f"Available backends: {e.available}")

BackendNotSupportedError: Backend 'ibm_quantum_2000' is not supported. Available: ibm_fez, ibm_torino, ibm_marrakesh, rigetti_ankaa, ionq_aria, ionq_forte, iqm_garnet, iqm_emerald, quantinuum_h2

Requested backend: ibm_quantum_2000
Available backends: ['ibm_fez', 'ibm_torino', 'ibm_marrakesh', 'rigetti_ankaa', 'ionq_aria', 'ionq_forte', 'iqm_garnet', 'iqm_emerald', 'quantinuum_h2']


In [6]:
# Also raised by QBCompiler.from_backend()
try:
    compiler = QBCompiler.from_backend("google_sycamore")
except BackendNotSupportedError as e:
    print(f"BackendNotSupportedError: {e}")

BackendNotSupportedError: Backend 'google_sycamore' is not supported. Available: ibm_fez, ibm_torino, ibm_marrakesh, rigetti_ankaa, ionq_aria, ionq_forte, iqm_garnet, iqm_emerald, quantinuum_h2


## 4. BudgetExceededError

Raised by `BudgetAwareStrategy` when the estimated execution cost
exceeds the caller's budget and cannot be reduced (even 1 shot is
too expensive).

In [7]:
from qb_compiler.strategies import BudgetAwareStrategy

# IonQ Aria: $0.30/shot.  Budget: $0.10.  Even 1 shot costs $0.30.
try:
    strategy = BudgetAwareStrategy(budget_usd=0.10, shots=100)
    ionq_config = CompilerConfig(backend="ionq_aria", optimization_level=2)
    strategy.build_pass_manager(ionq_config)
except BudgetExceededError as e:
    print(f"BudgetExceededError: {e}")
    print(f"\nEstimated cost: ${e.estimated_usd:.4f}")
    print(f"Budget:         ${e.budget_usd:.4f}")
    print(f"Shots:          {e.shots}")

## 5. CalibrationStaleError and CalibrationNotFoundError

These are raised when calibration data is too old or missing entirely.

In [8]:
# CalibrationStaleError: construct directly to demonstrate attributes
stale_err = CalibrationStaleError(
    backend="ibm_fez",
    age_hours=48.5,
    max_hours=24.0,
)
print(f"CalibrationStaleError: {stale_err}")
print(f"  backend:    {stale_err.backend}")
print(f"  age_hours:  {stale_err.age_hours}")
print(f"  max_hours:  {stale_err.max_hours}")

# CalibrationNotFoundError
not_found_err = CalibrationNotFoundError(backend="ibm_fez")
print(f"\nCalibrationNotFoundError: {not_found_err}")
print(f"  backend: {not_found_err.backend}")

CalibrationStaleError: Calibration for ibm_fez is 48.5h old (max allowed: 24.0h)
  backend:    ibm_fez
  age_hours:  48.5
  max_hours:  24.0

CalibrationNotFoundError: No calibration data found for backend 'ibm_fez'
  backend: ibm_fez


### Catching calibration errors generically

Since both inherit from `CalibrationError`, you can catch the parent
to handle all calibration-related problems.

In [9]:
def handle_calibration(err: CalibrationError) -> str:
    """Generic calibration error handler."""
    if isinstance(err, CalibrationStaleError):
        return f"Stale calibration for {err.backend} ({err.age_hours:.1f}h old). Refresh needed."
    elif isinstance(err, CalibrationNotFoundError):
        return f"No calibration for {err.backend}. Run calibration fetch first."
    else:
        return f"Calibration error: {err}"

# Test with both error types
for err in [stale_err, not_found_err]:
    result = handle_calibration(err)
    print(f"  -> {result}")

  -> Stale calibration for ibm_fez (48.5h old). Refresh needed.
  -> No calibration for ibm_fez. Run calibration fetch first.


## 6. Graceful degradation patterns

A common pattern is to attempt calibration-aware compilation and fall
back to a simpler strategy if calibration is unavailable.

In [10]:
def compile_with_fallback(circuit, backend="ibm_fez"):
    """Attempt calibration-aware compilation, fall back to speed strategy."""
    try:
        compiler = QBCompiler.from_backend(backend)
        result = compiler.compile(circuit, strategy="fidelity_optimal")
        print(f"Compiled with calibration: depth={result.compiled_depth}, "
              f"fidelity={result.estimated_fidelity:.4f}")
        return result

    except CalibrationError as e:
        print(f"Calibration issue: {e}")
        print("Falling back to speed_optimal strategy (no calibration)...")

        compiler = QBCompiler.from_backend(backend)
        result = compiler.compile(circuit, strategy="speed_optimal")
        print(f"Fallback result: depth={result.compiled_depth}")
        return result

circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()
compile_with_fallback(circ)

Compiled with calibration: depth=11, fidelity=0.9742


CompileResult(compiled_circuit=QBCircuit(n_qubits=24, ops=20, depth=11), original_depth=4, compiled_depth=11, estimated_fidelity=0.9741735836465424, pass_log=(PassResult(pass_name='basis_translation', elapsed_ms=0.115, depth_before=3, depth_after=11, gate_count_before=6, gate_count_after=20, detail=''), PassResult(pass_name='gate_cancellation', elapsed_ms=0.062, depth_before=11, depth_after=11, gate_count_before=20, gate_count_after=20, detail=''), PassResult(pass_name='rotation_merge', elapsed_ms=0.125, depth_before=11, depth_after=11, gate_count_before=20, gate_count_after=20, detail=''), PassResult(pass_name='gate_cancellation', elapsed_ms=0.035, depth_before=11, depth_after=11, gate_count_before=20, gate_count_after=20, detail=''), PassResult(pass_name='rotation_merge', elapsed_ms=0.131, depth_before=11, depth_after=11, gate_count_before=20, gate_count_after=20, detail='')), compilation_time_ms=9635.831, initial_layout={1: 23, 0: 22, 2: 16})

## 7. Retry strategies for transient errors

Some errors are transient (e.g., stale calibration that can be refreshed).
A retry pattern with exponential backoff can handle these.

In [11]:
import time

def compile_with_retry(
    circuit,
    backend="ibm_fez",
    max_retries=3,
    base_delay=1.0,
):
    """Compile with retry for transient calibration errors."""
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            compiler = QBCompiler.from_backend(backend)
            result = compiler.compile(circuit, strategy="fidelity_optimal")
            if attempt > 0:
                print(f"  Succeeded on attempt {attempt + 1}")
            return result

        except CalibrationStaleError as e:
            last_error = e
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"  Attempt {attempt + 1}: stale calibration "
                      f"({e.age_hours:.1f}h old). Retrying in {delay:.1f}s...")
                time.sleep(delay)
                # In production, you would refresh calibration here:
                # calibration_provider.refresh(backend)
            else:
                print(f"  All {max_retries + 1} attempts failed.")
                raise

        except (BackendNotSupportedError, InvalidCircuitError):
            # Non-retryable errors: fail immediately
            raise


# This will succeed on first attempt (calibration is fine)
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()
result = compile_with_retry(circ)
print(f"Result: depth={result.compiled_depth}")

Result: depth=11


## 8. Defensive coding patterns

Patterns for building robust applications with qb-compiler.

In [12]:
# Pattern 1: Validate backend before compilation
from qb_compiler.config import BACKEND_CONFIGS

def validate_backend(backend: str) -> bool:
    """Check if a backend is supported before attempting compilation."""
    return backend in BACKEND_CONFIGS

for b in ["ibm_fez", "ibm_torino", "ibm_quantum_2000", "google_sycamore"]:
    status = "supported" if validate_backend(b) else "NOT supported"
    print(f"  {b}: {status}")

  ibm_fez: supported
  ibm_torino: supported
  ibm_quantum_2000: NOT supported
  google_sycamore: NOT supported


In [13]:
# Pattern 2: Circuit validation before compile
def validate_circuit(circuit: QBCircuit, max_qubits: int = 50) -> list[str]:
    """Pre-validate a circuit and return a list of warnings/errors."""
    issues = []

    if circuit.gate_count == 0:
        issues.append("WARNING: Empty circuit (no gates)")

    if circuit.n_qubits > max_qubits:
        issues.append(f"WARNING: {circuit.n_qubits} qubits exceeds recommended max ({max_qubits})")

    if circuit.two_qubit_count == 0 and circuit.gate_count > 0:
        issues.append("INFO: No 2-qubit gates. Layout selection will have minimal impact.")

    depth_ratio = circuit.depth / max(circuit.gate_count, 1)
    if depth_ratio > 0.8:
        issues.append("INFO: High depth-to-gate ratio. Circuit has little parallelism.")

    return issues

# Test with various circuits
test_circuits = {
    "normal":   QBCircuit(5).h(0).cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4).measure_all(),
    "empty":    QBCircuit(3),
    "1Q only":  QBCircuit(3).h(0).h(1).h(2),
}

for name, circ in test_circuits.items():
    issues = validate_circuit(circ)
    print(f"\n{name} ({circ}):")
    if issues:
        for issue in issues:
            print(f"  {issue}")
    else:
        print("  All checks passed.")


normal (QBCircuit(n_qubits=5, ops=10, depth=6)):
  All checks passed.

empty (QBCircuit(n_qubits=3, ops=0, depth=0)):

1Q only (QBCircuit(n_qubits=3, ops=3, depth=1)):
  INFO: No 2-qubit gates. Layout selection will have minimal impact.


In [14]:
# Pattern 3: Budget guard wrapper
from qb_compiler.config import get_backend_spec

def estimate_cost(backend: str, shots: int, circuit_depth: int = 100) -> float:
    """Estimate execution cost in USD before committing to a run."""
    spec = get_backend_spec(backend)
    depth_factor = max(1.0, circuit_depth / 100.0)
    return spec.cost_per_shot * depth_factor * shots

def safe_compile(circuit, backend, shots=4096, max_budget_usd=50.0):
    """Compile with an automatic cost guard."""
    est = estimate_cost(backend, shots, circuit.depth)
    if est > max_budget_usd:
        raise BudgetExceededError(
            estimated_usd=est,
            budget_usd=max_budget_usd,
            shots=shots,
        )

    compiler = QBCompiler.from_backend(backend)
    return compiler.compile(circuit, budget_usd=max_budget_usd)

circ = QBCircuit(5).h(0).cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4).measure_all()

# Safe on IBM Fez
try:
    result = safe_compile(circ, "ibm_fez", shots=10_000)
    print(f"IBM Fez: compiled OK (est ${estimate_cost('ibm_fez', 10_000):.4f})")
except BudgetExceededError as e:
    print(f"IBM Fez: {e}")

# Would blow budget on IonQ Aria
try:
    result = safe_compile(circ, "ionq_aria", shots=10_000)
    print(f"IonQ Aria: compiled OK")
except BudgetExceededError as e:
    print(f"IonQ Aria: {e}")

IBM Fez: compiled OK (est $1.6000)
IonQ Aria: Estimated cost $3000.0000 exceeds budget $50.0000 (10000 shots)


## 9. Building robust pipelines with proper error handling

Putting it all together: a production-quality compilation pipeline
that handles all error cases.

In [15]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class PipelineResult:
    """Result from the robust compilation pipeline."""
    success: bool
    compile_result: Optional[object] = None
    error_type: Optional[str] = None
    error_message: Optional[str] = None
    fallback_used: bool = False


def robust_compile(circuit, backend, shots=4096, budget_usd=100.0):
    """Full production compilation pipeline with error handling."""

    # Step 1: Validate inputs
    if not validate_backend(backend):
        return PipelineResult(
            success=False,
            error_type="BackendNotSupportedError",
            error_message=f"Backend '{backend}' is not supported.",
        )

    issues = validate_circuit(circuit)
    for issue in issues:
        if issue.startswith("ERROR"):
            return PipelineResult(
                success=False,
                error_type="InvalidCircuitError",
                error_message=issue,
            )

    # Step 2: Estimate cost
    try:
        est = estimate_cost(backend, shots, circuit.depth)
        if est > budget_usd:
            return PipelineResult(
                success=False,
                error_type="BudgetExceededError",
                error_message=f"Estimated ${est:.4f} exceeds budget ${budget_usd:.2f}",
            )
    except BackendNotSupportedError as e:
        return PipelineResult(
            success=False,
            error_type="BackendNotSupportedError",
            error_message=str(e),
        )

    # Step 3: Compile with fallback
    try:
        compiler = QBCompiler.from_backend(backend)
        result = compiler.compile(circuit, strategy="fidelity_optimal")
        return PipelineResult(success=True, compile_result=result)

    except CalibrationError as e:
        # Fallback: compile without calibration
        try:
            compiler = QBCompiler.from_backend(backend)
            result = compiler.compile(circuit, strategy="speed_optimal")
            return PipelineResult(
                success=True,
                compile_result=result,
                fallback_used=True,
            )
        except QBCompilerError as fallback_err:
            return PipelineResult(
                success=False,
                error_type=type(fallback_err).__name__,
                error_message=str(fallback_err),
            )

    except QBCompilerError as e:
        return PipelineResult(
            success=False,
            error_type=type(e).__name__,
            error_message=str(e),
        )


# Test the pipeline with various scenarios
circ = QBCircuit(5).h(0).cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4).measure_all()

scenarios = [
    ("Normal", circ, "ibm_fez", 4096, 100.0),
    ("Unknown backend", circ, "ibm_phantom", 4096, 100.0),
    ("Over budget", circ, "ionq_aria", 1_000_000, 50.0),
]

for name, c, backend, shots, budget in scenarios:
    result = robust_compile(c, backend, shots, budget)
    status = "OK" if result.success else f"FAIL ({result.error_type})"
    extra = " [fallback]" if result.fallback_used else ""
    print(f"{name:>20}: {status}{extra}")
    if not result.success:
        print(f"{'':>20}  {result.error_message}")

              Normal: OK
     Unknown backend: FAIL (BackendNotSupportedError)
                      Backend 'ibm_phantom' is not supported.
         Over budget: FAIL (BudgetExceededError)
                      Estimated $300000.0000 exceeds budget $50.00


## 10. Blanket exception handling

For simple scripts, catching `QBCompilerError` covers all qb-compiler
exceptions in one handler.

In [16]:
def simple_compile(circuit, backend):
    """Simple compile with blanket error handling."""
    try:
        compiler = QBCompiler.from_backend(backend)
        return compiler.compile(circuit)
    except QBCompilerError as e:
        print(f"Compilation failed: [{type(e).__name__}] {e}")
        return None

# Success
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()
result = simple_compile(circ, "ibm_fez")
if result:
    print(f"Success: depth={result.compiled_depth}")

# Failure
result = simple_compile(circ, "nonexistent_backend")

Success: depth=11
Compilation failed: [BackendNotSupportedError] Backend 'nonexistent_backend' is not supported. Available: ibm_fez, ibm_torino, ibm_marrakesh, rigetti_ankaa, ionq_aria, ionq_forte, iqm_garnet, iqm_emerald, quantinuum_h2


## Summary

| Exception | When raised | Key attributes |
|-----------|------------|----------------|
| `QBCompilerError` | Base class (catch-all) | `message`, `detail` |
| `CompilationError` | General compilation failure | `message` |
| `InvalidCircuitError` | Malformed circuit | `gate`, `detail` |
| `CalibrationError` | Base for calibration issues | `message` |
| `CalibrationStaleError` | Calibration data too old | `backend`, `age_hours`, `max_hours` |
| `CalibrationNotFoundError` | No calibration data exists | `backend` |
| `BackendNotSupportedError` | Unknown backend name | `backend`, `available` |
| `BudgetExceededError` | Cost exceeds budget | `estimated_usd`, `budget_usd`, `shots` |

**Key patterns:**
- Catch `QBCompilerError` for blanket handling
- Catch specific exceptions for targeted recovery
- Use `CalibrationError` to catch both stale and missing calibration
- Pre-validate backends and circuits before compilation
- Implement cost guards before expensive operations
- Use fallback strategies when calibration is unavailable